# 6. Submission — загрузка индексов с сервера → submission.csv

**Порядок действий:**
1. Запустить пайплайн на H100 (`python -m kaggle.pipeline.run_pipeline`)
2. Упаковать индексы: `tar czf indexes.tar.gz /kaggle/working/*.index /kaggle/working/*.pkl /kaggle/working/scenes.jsonl /kaggle/working/events.jsonl`
3. Загрузить `indexes.tar.gz` как Kaggle Dataset (UI → New Dataset)
4. Подключить датасет к этому ноутбуку и запустить все ячейки

In [ ]:
# Зависимости
!pip install FlagEmbedding faiss-gpu-cu12 rank-bm25 -q

In [ ]:
import os, sys, json, pickle, tarfile
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from tqdm import tqdm
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from FlagEmbedding import BGEM3FlagModel, FlagReranker

In [ ]:
# ── Пути ────────────────────────────────────────────────────────────────────
# Данные соревнования
COMP_DIR = Path('/kaggle/input/competitions/multi-lingual-video-fragment-retrieval-challenge/video-rag')
TEST_CSV = COMP_DIR / 'test/test.csv'

# Индексы: ищем сначала в загруженном датасете, потом в /kaggle/working
# Если загрузили как датасет с именем 'video-rag-indexes':
DATASET_DIR = Path('/kaggle/input/video-rag-indexes')
WORK_DIR    = Path('/kaggle/working')

def find_indexes_dir() -> Path:
    """Найти папку с индексами — датасет или working dir."""
    # Если датасет подключён и там есть tar
    if DATASET_DIR.exists():
        tars = list(DATASET_DIR.glob('*.tar.gz')) + list(DATASET_DIR.glob('*.tar'))
        if tars:
            print(f'Распаковываем {tars[0]}...')
            with tarfile.open(tars[0]) as tf:
                tf.extractall(WORK_DIR)
            print('Распаковано.')
        # Если файлы лежат прямо в папке датасета
        if (DATASET_DIR / 'faiss_scenes.index').exists():
            return DATASET_DIR
    # Fallback: индексы уже в /kaggle/working
    return WORK_DIR

IDX_DIR = find_indexes_dir()
print(f'Индексы: {IDX_DIR}')
print('Файлы:', sorted(f.name for f in IDX_DIR.iterdir() if f.suffix in ('.index', '.pkl', '.jsonl')))

In [ ]:
# ── Параметры поиска ────────────────────────────────────────────────────────
BGE_MODEL      = 'BAAI/bge-m3'
RERANKER_MODEL = 'BAAI/bge-reranker-v2-m3'

TOP_K_DENSE    = 50
TOP_K_SPARSE   = 50
RRF_K          = 60
RERANKER_TOP_K = 100
FINAL_TOP_N    = 5

In [ ]:
# ── Загрузка индексов ───────────────────────────────────────────────────────
def load_pkl(path): 
    with open(path, 'rb') as f: return pickle.load(f)

print('Загружаем FAISS индексы...')
faiss_scenes = faiss.read_index(str(IDX_DIR / 'faiss_scenes.index'))
faiss_events = faiss.read_index(str(IDX_DIR / 'faiss_events.index'))

# Переносим на GPU для быстрого поиска
res = faiss.StandardGpuResources()
faiss_scenes = faiss.index_cpu_to_gpu(res, 0, faiss_scenes)
faiss_events = faiss.index_cpu_to_gpu(res, 0, faiss_events)
print(f'  scenes: {faiss_scenes.ntotal} векторов, events: {faiss_events.ntotal} векторов')

print('Загружаем метаданные и BM25...')
scenes_meta   = load_pkl(IDX_DIR / 'scenes_meta.pkl')
events_meta   = load_pkl(IDX_DIR / 'events_meta.pkl')
sparse_scenes = load_pkl(IDX_DIR / 'sparse_scenes.pkl')
sparse_events = load_pkl(IDX_DIR / 'sparse_events.pkl')
bm25_scenes   = load_pkl(IDX_DIR / 'bm25_scenes.pkl')
bm25_events   = load_pkl(IDX_DIR / 'bm25_events.pkl')
print(f'  scenes_meta: {len(scenes_meta)}, events_meta: {len(events_meta)}')

# Scene lookup для event → scene resolution
scene_lookup = {}
scenes_jsonl = IDX_DIR / 'scenes.jsonl'
if scenes_jsonl.exists():
    with open(scenes_jsonl) as f:
        for line in f:
            doc = json.loads(line.strip())
            scene_lookup[f"{doc['video_id']}__{doc['scene_idx']}"] = doc
print(f'  scene_lookup: {len(scene_lookup)} сцен')

In [ ]:
# ── Модели ──────────────────────────────────────────────────────────────────
print(f'Загружаем {BGE_MODEL}...')
bge_model = BGEM3FlagModel(BGE_MODEL, use_fp16=True)

print(f'Загружаем {RERANKER_MODEL}...')
reranker = FlagReranker(RERANKER_MODEL, use_fp16=True)
print('Готово.')

In [ ]:
# ── Query preprocessor ──────────────────────────────────────────────────────
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('/kaggle/working').parent))  # или путь к репо

from preproc.query_preprocessor import QueryPreprocessor
_train_csv = COMP_DIR / 'train' / 'train_qa.csv'
qp = QueryPreprocessor(str(_train_csv) if _train_csv.exists() else None, use_sage=False)
print('Query preprocessor готов.')

In [ ]:
# ── Вспомогательные функции ─────────────────────────────────────────────────
def iou(s1, e1, s2, e2):
    inter = max(0.0, min(e1, e2) - max(s1, s2))
    union = (e1 - s1) + (e2 - s2) - inter
    return inter / union if union > 0 else 0.0

def rrf_merge(ranked_lists, k=RRF_K):
    scores, items = defaultdict(float), {}
    for lst in ranked_lists:
        for rank, item in enumerate(lst, 1):
            uid = f"{item['video_id']}_{item['start']:.3f}_{item['end']:.3f}"
            scores[uid] += 1.0 / (k + rank)
            items.setdefault(uid, item)
    merged = sorted(items.values(), key=lambda x: scores[f"{x['video_id']}_{x['start']:.3f}_{x['end']:.3f}"], reverse=True)
    for item in merged:
        item['rrf_score'] = scores[f"{item['video_id']}_{item['start']:.3f}_{item['end']:.3f}"]
    return merged

def dedup(results, threshold=0.5):
    kept = []
    for c in results:
        if not any(c['video_id'] == e['video_id'] and iou(c['start'], c['end'], e['start'], e['end']) > threshold for e in kept):
            kept.append(c)
    return kept

def sparse_dot(qsparse, doc_list, top_k):
    scores = [(i, sum(qsparse.get(k, 0.0) * v for k, v in doc.items())) for i, doc in enumerate(doc_list)]
    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

def encode_query(query):
    out = bge_model.encode([query], return_dense=True, return_sparse=True)
    dense  = out['dense_vecs'][0]
    sparse = {str(k): float(v) for k, v in out['lexical_weights'][0].items()}
    return dense, sparse

def faiss_search(index, meta, vec, top_k, label):
    scores, indices = index.search(vec.reshape(1, -1).astype(np.float32), top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1: continue
        doc = meta[idx]
        results.append({'video_id': doc['video_id'], 'start': doc.get('start', 0.0), 'end': doc.get('end', 0.0),
                        'scene_idx': doc.get('scene_idx'), 'center_scene_idx': doc.get('center_scene_idx'),
                        'text': doc.get('summary', '') or doc.get('asr_text', '') or doc.get('event_summary', ''),
                        'source': label})
    return results

def bm25_search(bm25, meta, query, top_k, label):
    scores = bm25.get_scores(query.lower().split())
    results = []
    for idx in np.argsort(scores)[::-1][:top_k]:
        if scores[idx] <= 0: break
        doc = meta[idx]
        results.append({'video_id': doc['video_id'], 'start': doc.get('start', 0.0), 'end': doc.get('end', 0.0),
                        'scene_idx': doc.get('scene_idx'), 'center_scene_idx': doc.get('center_scene_idx'),
                        'text': doc.get('asr_text', '') or doc.get('summary', '') or doc.get('event_summary', ''),
                        'source': label})
    return results

def sparse_search(qsparse, doc_list, meta, top_k, label):
    results = []
    for idx, score in sparse_dot(qsparse, doc_list, top_k):
        if score <= 0: break
        doc = meta[idx]
        results.append({'video_id': doc['video_id'], 'start': doc.get('start', 0.0), 'end': doc.get('end', 0.0),
                        'scene_idx': doc.get('scene_idx'), 'center_scene_idx': doc.get('center_scene_idx'),
                        'text': doc.get('asr_text', '') or doc.get('summary', '') or doc.get('event_summary', ''),
                        'source': label})
    return results

def resolve_event(r):
    center = r.get('center_scene_idx')
    if center is None: return r
    scene = scene_lookup.get(f"{r['video_id']}__{center}")
    if scene:
        r = r.copy()
        r['start'], r['end'] = scene['start'], scene['end']
    return r

In [ ]:
# ── Основная функция поиска ──────────────────────────────────────────────────
def search(query, top_n=FINAL_TOP_N):
    clean_query = qp(query)
    dense_vec, sparse_vec = encode_query(clean_query)

    with ThreadPoolExecutor(max_workers=6) as pool:
        futures = [
            pool.submit(faiss_search,  faiss_scenes, scenes_meta, dense_vec,  TOP_K_DENSE,   'dense_scenes'),
            pool.submit(faiss_search,  faiss_events, events_meta, dense_vec,  TOP_K_DENSE,   'dense_events'),
            pool.submit(bm25_search,   bm25_scenes,  scenes_meta, clean_query,      TOP_K_SPARSE,  'bm25_scenes'),
            pool.submit(bm25_search,   bm25_events,  events_meta, clean_query,      TOP_K_SPARSE,  'bm25_events'),
            pool.submit(sparse_search, sparse_vec,   sparse_scenes, scenes_meta, TOP_K_SPARSE, 'sparse_scenes'),
            pool.submit(sparse_search, sparse_vec,   sparse_events, events_meta, TOP_K_SPARSE, 'sparse_events'),
        ]
        ranked_lists = [f.result() for f in as_completed(futures)]

    candidates = dedup(rrf_merge(ranked_lists))[:RERANKER_TOP_K]

    if candidates:
        pairs = [[clean_query, c.get('text', '')] for c in candidates]
        rscores = reranker.compute_score(pairs)
        if isinstance(rscores, float): rscores = [rscores]
        for c, s in zip(candidates, rscores):
            c['reranker_score'] = float(s)
        candidates.sort(key=lambda x: x['reranker_score'], reverse=True)

    return [resolve_event(c) for c in candidates[:top_n]]

# Быстрый тест
test_results = search('car chase at night')
for r in test_results:
    print(f"  {r['video_id']}  {r['start']:.1f}s – {r['end']:.1f}s  score={r.get('reranker_score', 0):.3f}")

In [ ]:
# ── Генерация submission.csv ─────────────────────────────────────────────────
test_df = pd.read_csv(TEST_CSV)
print(f'Запросов: {len(test_df)}')
print(test_df.head(3))

rows = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Поиск'):
    qid     = row['query_id']
    query   = row['question']
    results = search(query, top_n=FINAL_TOP_N)

    out = {'query_id': qid}
    for rank in range(1, FINAL_TOP_N + 1):
        if rank <= len(results):
            r = results[rank - 1]
            out[f'video_file_{rank}'] = r['video_id']
            out[f'start_{rank}']      = round(r['start'], 3)
            out[f'end_{rank}']        = round(r['end'], 3)
        else:
            out[f'video_file_{rank}'] = ''
            out[f'start_{rank}']      = 0.0
            out[f'end_{rank}']        = 0.0
    rows.append(out)

cols = ['query_id']
for rank in range(1, FINAL_TOP_N + 1):
    cols += [f'video_file_{rank}', f'start_{rank}', f'end_{rank}']

submission_df = pd.DataFrame(rows)[cols]
submission_df.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Сохранено: /kaggle/working/submission.csv — {len(submission_df)} строк')
print(submission_df.head(3))

In [ ]:
# ── Проверка формата ─────────────────────────────────────────────────────────
assert list(submission_df.columns) == cols, 'Неправильные колонки!'
assert len(submission_df) == len(test_df), f'Строк {len(submission_df)} != запросов {len(test_df)}'
filled = submission_df['video_file_1'].str.match(r'^video_[a-f0-9]+$').mean()
print(f'Заполненность video_file_1: {filled:.1%}')
assert filled > 0.9, 'Слишком много пустых результатов!'
print('Проверка пройдена. Можно сабмитить!')